In [ ]:
"""
Preprocess Open Ephys electrophysiology recordings for LFP and spike-band analysis.

Expected repository structure:

DATA/
└── ephys/
    ├── mouse_id/
    │   ├── Record Node open ephys/
    │   ├── exploration_segments.csv
    │   └── LED_info.csv
    └── mouse_id/
        ├── Record Node open ephys/
        ├── exploration_segments.csv
        └── LED_info.csv

For each recording:
- `Record Node open ephys/` contains the raw Open Ephys recording.
- `exploration_segments.csv` contains the start/end frames and timestamps
  of the behavioral exploration segments.
- `LED_info.csv` contains the frame corresponding to the LED synchronization
  event and is used to align the electrophysiological recording.
"""

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.widgets as sw
import sys

REPO_ROOT = Path("..").resolve()
sys.path.append(str(REPO_ROOT))

In [ ]:
# Local path to the Open Ephys recording
RAW_DATA_DIR = Path(
    "DATA/ephys/mouse_id/Record Node 112"
)

# Open Ephys block containing the recording
OPEN_EPHYS_BLOCK = 1

# Output directory for the processed recording
OUTPUT_DIR = RAW_DATA_DIR.parent.parent / "mouse_id"


# ADC channel used for LED synchronization
LED_CHANNEL = "ADC1"
SEGM_INDEX = 0
# LED threshold used to detect the synchronization signal
LED_THRESHOLD_MV = 4000
LED_THRESHOLD_UV = LED_THRESHOLD_MV * 1000

LFP_FILTER_MIN_HZ = 0.5
LFP_FILTER_MAX_HZ = 150.0
LFP_RESAMPLE_HZ = 1000

SPIKE_FILTER_MIN_HZ = 600.0
SPIKE_FILTER_MAX_HZ = 6000.0
CAR_ALPHA = 1 # Tune CAR subtraction to reduce common noise without removing shared neural activity


QC_TIME_RANGE_S = (30, 40)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [ ]:
recording = se.OpenEphysBinaryRecordingExtractor(
    RAW_DATA_DIR,
    block_index=OPEN_EPHYS_BLOCK,
)

sampling_frequency = float(
    recording.get_sampling_frequency()
)

n_samples = recording.get_num_frames(
    segment_index= SEGM_INDEX
)

duration_s = n_samples / sampling_frequency

print(f"Sampling frequency: {sampling_frequency:.2f} Hz")
print(f"Number of samples: {n_samples:,}")
print(f"Duration: {duration_s / 60:.2f} min")
print(f"Number of channels: {recording.get_num_channels()}")
print(f"Channels: {recording.channel_ids}")

In [ ]:
if LED_CHANNEL not in recording.channel_ids:
    raise ValueError(
        f"Synchronization channel '{LED_CHANNEL}' "
        "was not found in the recording."
    )

print(
    f"Synchronization channel '{LED_CHANNEL}' found."
)

In [ ]:
# Detect the first LED synchronization sample above the threshold

led_trace = recording.get_traces(
    segment_index=SEGM_INDEX,
    channel_ids=[LED_CHANNEL],
    return_scaled=True,
).squeeze()

sync_candidates = np.flatnonzero(
    led_trace > LED_THRESHOLD_UV
)

if len(sync_candidates) == 0:
    raise RuntimeError(
        f"No synchronization signal was detected on "
        f"{LED_CHANNEL} above threshold "
        f"{LED_THRESHOLD_UV}."
    )

led_sync_sample = int(
    sync_candidates[0]
)

led_sync_time_s = (
    led_sync_sample / sampling_frequency
)

print(
    f"LED synchronization detected at sample "
    f"{led_sync_sample:,} "
    f"({led_sync_time_s:.3f} s)."
)

In [ ]:
# Trim the recording to start at the LED synchronization event

synchronized_traces = recording.get_traces(
    segment_index=SEGM_INDEX,
    start_frame=led_sync_sample,
    end_frame=n_samples,
    return_scaled=True,
)

synchronized_recording = si.NumpyRecording(
    traces_list=[synchronized_traces],
    sampling_frequency=sampling_frequency,
    channel_ids=recording.channel_ids,
)

print(
    f"Synchronized duration: "
    f"{synchronized_recording.get_total_duration():.2f} s"
)

In [ ]:

neural_channels = [
    channel
    for channel in synchronized_recording.channel_ids
    if not str(channel).startswith("ADC")
]

print(
    f"Neural channels available for QC: "
    f"{len(neural_channels)}"
)

In [ ]:
# Visual inspection of neural channels for quality control

sw.plot_traces(
    synchronized_recording.channel_slice(
        channel_ids=neural_channels
    ),
    show_channel_ids=True,
    segment_index=0,
    time_range=QC_TIME_RANGE_S,
)

In [ ]:
# Channels excluded from downstream analysis based on visual QC

BAD_CHANNELS = [
    "CH5",
    "CH6",
    "CH7",
    "CH8",
    "CH12",
]

In [ ]:
analysis_channels = [
    channel
    for channel in synchronized_recording.channel_ids
    if channel not in BAD_CHANNELS
    and not str(channel).startswith("ADC")
]

processed_recording = (
    synchronized_recording.channel_slice(
        channel_ids=analysis_channels
    )
)

print(
    f"Remaining analysis channels: "
    f"{processed_recording.channel_ids}"
)

In [ ]:
# PFC electrode channels
PFC_CHANNELS = [
    "CH1",
    "CH2",
    "CH3",
    "CH4",
    "CH13",
    "CH14",
    "CH15",
    "CH16",
]

# RSC electrode channels
RSC_CHANNELS = [
    "CH9",
    "CH10",
    "CH11",
]

In [ ]:
# LFP preprocessing: bandpass filter and downsample to 1 kHz

lfp_recording = spre.bandpass_filter(
    processed_recording,
    freq_min=LFP_FILTER_MIN_HZ,
    freq_max=LFP_FILTER_MAX_HZ,
)

lfp_recording = spre.resample(
    lfp_recording,
    resample_rate=LFP_RESAMPLE_HZ,
)

print(
    f"LFP sampling frequency: "
    f"{lfp_recording.get_sampling_frequency():.1f} Hz"
)

sw.plot_traces(
    lfp_recording,
    show_channel_ids=True,
    segment_index=0,
    time_range=QC_TIME_RANGE_S,
)

In [ ]:
# Apply common-average referencing using the median across channels

spike_recording = spre.bandpass_filter(
    processed_recording,
    freq_min=SPIKE_FILTER_MIN_HZ,
    freq_max=SPIKE_FILTER_MAX_HZ,
)

spike_traces = spike_recording.get_traces()

spike_data = pd.DataFrame(
    spike_traces,
    columns=spike_recording.channel_ids,
)

median_reference = spike_data.median(axis=1)

spike_data = spike_data.subtract(
    CAR_ALPHA * median_reference,
    axis=0,
)

spike_recording = si.NumpyRecording(
    traces_list=[
        spike_data.to_numpy(dtype=np.float32)
    ],
    sampling_frequency=(
        spike_recording.get_sampling_frequency()
    ),
    channel_ids=spike_recording.channel_ids,
)

print(
    f"Spike-band sampling frequency: "
    f"{spike_recording.get_sampling_frequency():.1f} Hz"
)

sw.plot_traces(
    spike_recording,
    show_channel_ids=True,
    segment_index=0,
    time_range=QC_TIME_RANGE_S,
)

In [ ]:
# Record preprocessing parameters and metadata for reproducibility

metadata = {
    "open_ephys_block_index": OPEN_EPHYS_BLOCK,

    "synchronization": {
        "channel": LED_CHANNEL,
        "threshold": LED_THRESHOLD_UV,
        "start_sample": led_sync_sample,
        "start_time_s": led_sync_time_s,
    },

    "quality_control": {
        "excluded_channels": BAD_CHANNELS,
    },

    "anatomical_assignment": {
        "pfc_channels": PFC_CHANNELS,
        "rsc_channels": RSC_CHANNELS,
    },

    "lfp_processing": {
        "filter": "bandpass",
        "fmin_hz": LFP_FILTER_MIN_HZ,
        "fmax_hz": LFP_FILTER_MAX_HZ,
        "resample_hz": LFP_RESAMPLE_HZ,
    },

    "spike_processing": {
        "filter": "bandpass",
        "fmin_hz": SPIKE_FILTER_MIN_HZ,
        "fmax_hz": SPIKE_FILTER_MAX_HZ,
        "reference": "median",
        "reference_coefficient": CAR_ALPHA,
    },
}

In [ ]:
# Save processed LFP and spike-band recordings with their metadata

from utils.io import save_processed_recording

lfp_path = save_processed_recording(
    recording=lfp_recording,
    output_dir=OUTPUT_DIR,
    name="lfp",
    metadata=metadata,
)

spike_path = save_processed_recording(
    recording=spike_recording,
    output_dir=OUTPUT_DIR,
    name="spike",
    metadata=metadata,
)

print(f"LFP saved to: {lfp_path}")
print(f"Spike-band recording saved to: {spike_path}")